# Hybrid Recommendations

## 1. Data Preparation

In [1]:
from recsys_common import *

configure_notebook()

project_dir = Path.cwd()
shared_dir = project_dir / "artifacts" / "shared"

required_shared = [
    shared_dir / "df_model.csv",
    shared_dir / "train_df.csv",
    shared_dir / "test_df.csv",
]
missing_shared = [str(p) for p in required_shared if not p.exists()]
if missing_shared:
    raise FileNotFoundError(
        "Shared artifacts are required. Run 0_Data_Exploration.ipynb first. Missing:\\n"
        + "\\n".join(missing_shared)
    )

print("Loading prepared data from artifacts/shared/")
df_model = pd.read_csv(shared_dir / "df_model.csv")
train_df = pd.read_csv(shared_dir / "train_df.csv")
test_df = pd.read_csv(shared_dir / "test_df.csv")

for frame in (df_model, train_df, test_df):
    if "review_time" in frame.columns:
        frame["review_time"] = pd.to_datetime(frame["review_time"], errors="coerce")

print(f"Shared df_model: {df_model.shape}")
print(f"Shared train_df: {train_df.shape}")
print(f"Shared test_df: {test_df.shape}")


/Users/onosaeka/Uni/BCSAI/3-2/Chatbot&RecSys/Recommender-Sys/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading prepared data from artifacts/shared/
Shared df_model: (268743, 15)
Shared train_df: (155052, 15)
Shared test_df: (113691, 15)


In [2]:
print("Shared plotting imports and settings loaded from recsys_common.py")

Shared plotting imports and settings loaded from recsys_common.py


In [3]:
relevance_threshold = 4
df_model["is_relevant"] = (df_model["Score"] >= relevance_threshold).astype(int)

print("Relevant interaction rate:", round(df_model["is_relevant"].mean(), 4))

Relevant interaction rate: 0.7814


In [4]:
print("Using shared split loaded from artifacts/shared/")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train users:", train_df["UserId"].nunique())
print("Test users:", test_df["UserId"].nunique())
print("Train products:", train_df["ProductId"].nunique())
print("Test products:", test_df["ProductId"].nunique())

Using shared split loaded from artifacts/shared/
Train shape: (155052, 15)
Test shape: (113691, 15)
Train users: 17678
Test users: 79204
Train products: 14279
Test products: 34110


## 2. Evaluation Helper Functions

In [5]:
K = 10

print(f"Ranking cutoff K: {K}")
print(f"Relevance threshold: {relevance_threshold}")

Ranking cutoff K: 10
Relevance threshold: 4


In [6]:
print("Shared evaluation helpers loaded from recsys_common.py")

Shared evaluation helpers loaded from recsys_common.py


Evaluation note: ranking metrics here are computed on each user's held-out test interactions (not on full-catalog candidate ranking). This is acceptable for lightweight offline comparison, but values can be optimistic versus a full Top-N candidate evaluation setting.


################

## **9. Hybrid Recommendations**

### 9.1 Hybrid design

### 9.2 Score combination strategy

### 9.3 Hybrid evaluation

### 9.4 Hybrid Pipeline Overview

The hybrid recommender below is split into small blocks:
1. Load saved CF/CB/popularity artifacts
2. Normalize component scores
3. Apply weighted fusion with cold-start fallback
4. Generate Top-N recommendations
5. Compute evaluation and export artifacts

### 9.4 Load Saved Artifacts

Load outputs produced by the baselines/CF and content-based notebooks.

In [7]:

cf_dir = project_dir / "artifacts" / "baselines_cf"
cb_dir = project_dir / "artifacts" / "content_based"
hybrid_dir = project_dir / "artifacts" / "hybrid"
hybrid_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    cf_dir / "cf_test_scores.csv",
    cf_dir / "cf_topn_scores.csv",
    cf_dir / "pop_scores.csv",
    cb_dir / "cb_test_scores.csv",
    cb_dir / "cb_topn_scores.csv",
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing_files))

cf_test_scores = pd.read_csv(cf_dir / "cf_test_scores.csv")
cf_topn_scores = pd.read_csv(cf_dir / "cf_topn_scores.csv")
pop_scores = pd.read_csv(cf_dir / "pop_scores.csv")
cb_test_scores = pd.read_csv(cb_dir / "cb_test_scores.csv")
cb_topn_scores = pd.read_csv(cb_dir / "cb_topn_scores.csv")

print("Loaded artifact shapes:")
print("cf_test_scores:", cf_test_scores.shape)
print("cf_topn_scores:", cf_topn_scores.shape)
print("cb_test_scores:", cb_test_scores.shape)
print("cb_topn_scores:", cb_topn_scores.shape)
print("pop_scores:", pop_scores.shape)

Loaded artifact shapes:
cf_test_scores: (113691, 3)
cf_topn_scores: (3000, 3)
cb_test_scores: (53749, 4)
cb_topn_scores: (67, 3)
pop_scores: (14279, 2)


### 9.5 Configure Weights and Normalization

Use min-max normalized component scores before weighted fusion.

In [8]:
w_cf = 0.55
w_cb = 0.35
w_pop = 0.10
TOP_N = 10
MAX_CATALOG_USERS = 300

def minmax_bounds(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0.0, 1.0
    return float(s.min()), float(s.max())

def normalize_with_bounds(series, bounds):
    lo, hi = bounds
    s = pd.to_numeric(series, errors="coerce")
    if hi <= lo:
        return pd.Series(np.where(s.notna(), 0.5, np.nan), index=s.index, dtype=float)
    return (s - lo) / (hi - lo)

cf_bounds = minmax_bounds(pd.concat([cf_test_scores["cf_score"], cf_topn_scores["cf_score"]], axis=0))
cb_bounds = minmax_bounds(pd.concat([cb_test_scores["cb_score"], cb_topn_scores["cb_score"]], axis=0))
pop_bounds = minmax_bounds(pop_scores["pop_score"])

train_user_set = set(train_df["UserId"].astype(str))
train_item_set = set(train_df["ProductId"].astype(str))

### 9.6 Hybrid Score Function with Cold-Start Rules

In [9]:
def hybrid_score_from_row(row):
    user_known = row["UserId"] in train_user_set
    item_known = row["ProductId"] in train_item_set

    cf_n = row.get("cf_norm", np.nan)
    cb_n = row.get("cb_norm", np.nan)
    pop_n = row.get("pop_norm", np.nan)

    if (not user_known) and (not item_known):
        return pop_n if pd.notna(pop_n) else 0.5
    if not user_known:
        return pop_n if pd.notna(pop_n) else 0.5
    if not item_known:
        if pd.notna(cb_n):
            return cb_n
        return pop_n if pd.notna(pop_n) else 0.5

    terms = []
    if pd.notna(cf_n):
        terms.append((w_cf, cf_n))
    if pd.notna(cb_n):
        terms.append((w_cb, cb_n))
    if pd.notna(pop_n):
        terms.append((w_pop, pop_n))

    if not terms:
        return 0.5

    weight_sum = sum(w for w, _ in terms)
    return sum(w * s for w, s in terms) / weight_sum

### 9.7 Evaluate Hybrid on Test Interaction Pairs

In [10]:
hybrid_eval_df = test_df[["UserId", "ProductId", "Score"]].copy()
hybrid_eval_df["UserId"] = hybrid_eval_df["UserId"].astype(str)
hybrid_eval_df["ProductId"] = hybrid_eval_df["ProductId"].astype(str)

cf_test_tmp = cf_test_scores.copy()
cf_test_tmp["UserId"] = cf_test_tmp["UserId"].astype(str)
cf_test_tmp["ProductId"] = cf_test_tmp["ProductId"].astype(str)

cb_test_tmp = cb_test_scores.copy()
cb_test_tmp["UserId"] = cb_test_tmp["UserId"].astype(str)
cb_test_tmp["ProductId"] = cb_test_tmp["ProductId"].astype(str)

pop_tmp = pop_scores.copy()
pop_tmp["ProductId"] = pop_tmp["ProductId"].astype(str)

hybrid_eval_df = hybrid_eval_df.merge(cf_test_tmp[["UserId", "ProductId", "cf_score"]], on=["UserId", "ProductId"], how="left")
hybrid_eval_df = hybrid_eval_df.merge(cb_test_tmp[["UserId", "ProductId", "cb_score"]], on=["UserId", "ProductId"], how="left")
hybrid_eval_df = hybrid_eval_df.merge(pop_tmp[["ProductId", "pop_score"]], on="ProductId", how="left")

missing_cf = hybrid_eval_df["cf_score"].isna().mean()
missing_cb = hybrid_eval_df["cb_score"].isna().mean()
print(f"Missing cf_score rate on test pairs: {missing_cf:.2%}")
print(f"Missing cb_score rate on test pairs: {missing_cb:.2%}")

hybrid_eval_df["cf_norm"] = normalize_with_bounds(hybrid_eval_df["cf_score"], cf_bounds)
hybrid_eval_df["cb_norm"] = normalize_with_bounds(hybrid_eval_df["cb_score"], cb_bounds)
hybrid_eval_df["pop_norm"] = normalize_with_bounds(hybrid_eval_df["pop_score"], pop_bounds)

hybrid_eval_df["hybrid_norm_score"] = hybrid_eval_df.apply(hybrid_score_from_row, axis=1)
hybrid_eval_df["hybrid_score"] = 1.0 + 4.0 * hybrid_eval_df["hybrid_norm_score"]

hybrid_rmse = float(np.sqrt(mean_squared_error(hybrid_eval_df["Score"], hybrid_eval_df["hybrid_score"])))
hybrid_mae = float(mean_absolute_error(hybrid_eval_df["Score"], hybrid_eval_df["hybrid_score"]))

print("Hybrid test-pair metrics")
print(f"RMSE: {hybrid_rmse:.4f}")
print(f"MAE:  {hybrid_mae:.4f}")
print("Ranking metrics are computed on full-candidate Top-N lists below.")

Missing cf_score rate on test pairs: 0.00%
Missing cb_score rate on test pairs: 78.72%
Hybrid test-pair metrics
RMSE: 1.3531
MAE:  1.1226
Ranking metrics are computed on full-candidate Top-N lists below.


### 9.8 Build Hybrid Top-N Recommendations

In [11]:
cf_topn_tmp = cf_topn_scores.copy()
cf_topn_tmp["UserId"] = cf_topn_tmp["UserId"].astype(str)
cf_topn_tmp["ProductId"] = cf_topn_tmp["ProductId"].astype(str)

cb_topn_tmp = cb_topn_scores.copy()
cb_topn_tmp["UserId"] = cb_topn_tmp["UserId"].astype(str)
cb_topn_tmp["ProductId"] = cb_topn_tmp["ProductId"].astype(str)

# Build fast score lookups from upstream artifacts.
cf_lookup = {(u, i): float(s) for u, i, s in cf_topn_tmp[["UserId", "ProductId", "cf_score"]].itertuples(index=False, name=None)}
cb_lookup = {(u, i): float(s) for u, i, s in cb_topn_tmp[["UserId", "ProductId", "cb_score"]].itertuples(index=False, name=None)}
pop_lookup = {i: float(s) for i, s in pop_tmp[["ProductId", "pop_score"]].itertuples(index=False, name=None)}

def normalize_scalar(value, bounds):
    lo, hi = bounds
    if pd.isna(value):
        return np.nan
    if hi <= lo:
        return 0.5
    return (float(value) - lo) / (hi - lo)

train_seen_items = (
    train_df.assign(UserId=train_df["UserId"].astype(str), ProductId=train_df["ProductId"].astype(str))
    .groupby("UserId")["ProductId"]
    .apply(set)
    .to_dict()
 )

all_catalog_items = sorted(train_df["ProductId"].astype(str).unique())
candidate_users = sorted(set(test_df["UserId"].astype(str).unique()))
if MAX_CATALOG_USERS is not None:
    candidate_users = candidate_users[:MAX_CATALOG_USERS]

hybrid_topn_rows = []
candidate_export_rows = []
EXPORT_CANDIDATES_PER_USER = 200

for user_id in candidate_users:
    seen = train_seen_items.get(user_id, set())
    candidate_items = [item_id for item_id in all_catalog_items if item_id not in seen]

    if not candidate_items:
        continue

    user_rows = []
    for item_id in candidate_items:
        row = {
            "UserId": user_id,
            "ProductId": item_id,
            "cf_score": cf_lookup.get((user_id, item_id), np.nan),
            "cb_score": cb_lookup.get((user_id, item_id), np.nan),
            "pop_score": pop_lookup.get(item_id, np.nan),
        }
        row["cf_norm"] = normalize_scalar(row["cf_score"], cf_bounds)
        row["cb_norm"] = normalize_scalar(row["cb_score"], cb_bounds)
        row["pop_norm"] = normalize_scalar(row["pop_score"], pop_bounds)
        row["hybrid_norm_score"] = hybrid_score_from_row(row)
        row["hybrid_score"] = 1.0 + 4.0 * row["hybrid_norm_score"]
        user_rows.append(row)

    user_rows.sort(key=lambda x: x["hybrid_score"], reverse=True)

    for row in user_rows[:TOP_N]:
        hybrid_topn_rows.append((row["UserId"], row["ProductId"], row["hybrid_score"]))

    for row in user_rows[:EXPORT_CANDIDATES_PER_USER]:
        candidate_export_rows.append(
            (row["UserId"], row["ProductId"], row["cf_score"], row["cb_score"], row["pop_score"], row["hybrid_score"])
        )

hybrid_topn_df = pd.DataFrame(hybrid_topn_rows, columns=["UserId", "ProductId", "hybrid_score"])
candidate_scores = pd.DataFrame(
    candidate_export_rows,
    columns=["UserId", "ProductId", "cf_score", "cb_score", "pop_score", "hybrid_score"]
 )

print("Hybrid full-candidate Top-N rows:", len(hybrid_topn_df))
print("Export candidate rows:", len(candidate_scores))

# Realistic ranking metrics from full-candidate Top-N lists
K = 10 if "K" not in globals() else K
test_eval = test_df[["UserId", "ProductId", "Score"]].copy()
test_eval["UserId"] = test_eval["UserId"].astype(str)
test_eval["ProductId"] = test_eval["ProductId"].astype(str)

hybrid_topn = (
    hybrid_topn_df.sort_values(["UserId", "hybrid_score"], ascending=[True, False])
    .groupby("UserId")
    .apply(lambda g: list(g[["ProductId", "hybrid_score"]].itertuples(index=False, name=None)))
    .to_dict()
 )
hybrid_rank = ranking_metrics_from_topn(hybrid_topn, eval_df=test_eval, k=K, threshold=relevance_threshold)

print("Hybrid Ranking Metrics (full-candidate)")
print(f"Precision@{K}: {hybrid_rank['precision']:.4f}")
print(f"Recall@{K}: {hybrid_rank['recall']:.4f}")
print(f"MAP@{K}: {hybrid_rank['map']:.4f}")
print(f"NDCG@{K}: {hybrid_rank['ndcg']:.4f}")

Hybrid full-candidate Top-N rows: 3000
Export candidate rows: 60000
Hybrid Ranking Metrics (full-candidate)
Precision@10: 0.0000
Recall@10: 0.0000
MAP@10: 0.0000
NDCG@10: 0.0000


### 9.9 Compute Catalog Metrics

In [12]:
def coverage_at_n(topn_df, catalog_items):
    if len(catalog_items) == 0:
        return np.nan
    return topn_df["ProductId"].nunique() / len(catalog_items)

def personalization_at_n(topn_df, catalog_items):
    users = sorted(topn_df["UserId"].unique())
    if len(users) < 2:
        return np.nan
    item_to_col = {item_id: idx for idx, item_id in enumerate(catalog_items)}

    rows, cols, data = [], [], []
    for r, user_id in enumerate(users):
        user_items = topn_df[topn_df["UserId"] == user_id]["ProductId"].tolist()
        for item_id in user_items:
            if item_id in item_to_col:
                rows.append(r)
                cols.append(item_to_col[item_id])
                data.append(1)

    rec_matrix = csr_matrix((data, (rows, cols)), shape=(len(users), len(catalog_items)))
    sim_matrix = cosine_similarity(rec_matrix)
    upper = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    return float(1 - upper.mean()) if len(upper) > 0 else np.nan

def intra_list_similarity_at_n(topn_df, item_similarity_df, item_to_idx):
    ils_scores = []
    for _, group in topn_df.groupby("UserId"):
        rec_items = [item for item in group["ProductId"].tolist() if item in item_to_idx]
        if len(rec_items) < 2:
            continue
        idx = [item_to_idx[item_id] for item_id in rec_items]
        sim_block = item_similarity_df[idx][:, idx].toarray()
        upper = sim_block[np.triu_indices_from(sim_block, k=1)]
        if len(upper) > 0:
            ils_scores.append(float(upper.mean()))
    return float(np.mean(ils_scores)) if ils_scores else np.nan

all_train_items = sorted(train_df["ProductId"].astype(str).unique())

item_codes, item_labels = pd.factorize(train_df["ProductId"].astype(str))
user_codes, user_labels = pd.factorize(train_df["UserId"].astype(str))
item_user_matrix = csr_matrix(
    (train_df["Score"].astype(float), (item_codes, user_codes)),
    shape=(len(item_labels), len(user_labels))
)
item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_labels)}

hybrid_coverage = coverage_at_n(hybrid_topn_df, all_train_items)
hybrid_personalization = personalization_at_n(hybrid_topn_df, all_train_items)
hybrid_ils = intra_list_similarity_at_n(hybrid_topn_df, item_similarity, item_to_idx)

print("Hybrid catalog metrics")
print(f"Coverage: {hybrid_coverage:.4f}")
print(f"Personalization: {hybrid_personalization:.4f}")
print(f"Intra-list similarity: {hybrid_ils:.4f}")

Hybrid catalog metrics
Coverage: 0.0026
Personalization: 0.0529
Intra-list similarity: 0.3186


### 9.10 Save Hybrid Artifacts

In [13]:
# Fix: these metrics are stored in hybrid_rank (a dict),
# but the notebook tries to use hybrid_precision/hybrid_recall/... which may not exist.

if "hybrid_rank" not in globals():
    raise RuntimeError("hybrid_rank not found. Run Cell 25 (Build Hybrid Top-N Recommendations) first.")
if "hybrid_topn_df" not in globals():
    raise RuntimeError("hybrid_topn_df not found. Run Cell 25 first.")
if "candidate_scores" not in globals():
    raise RuntimeError("candidate_scores not found. Run Cell 25 first.")
if "hybrid_eval_df" not in globals():
    raise RuntimeError("hybrid_eval_df not found. Run the hybrid test-pair evaluation cell first.")
if "hybrid_dir" not in globals():
    raise RuntimeError("hybrid_dir not found. Run the artifact-loading cell first.")

hybrid_precision = float(hybrid_rank["precision"])
hybrid_recall = float(hybrid_rank["recall"])
hybrid_map = float(hybrid_rank["map"])
hybrid_ndcg = float(hybrid_rank["ndcg"])

hybrid_eval_summary = pd.DataFrame([
    {
        "Model": "Weighted Hybrid (CF+CB+Popularity)",
        "w_cf": w_cf,
        "w_cb": w_cb,
        "w_pop": w_pop,
        "RMSE": hybrid_rmse,
        "MAE": hybrid_mae,
        f"Precision@{K}": hybrid_precision,
        f"Recall@{K}": hybrid_recall,
        f"MAP@{K}": hybrid_map,
        f"NDCG@{K}": hybrid_ndcg,
        "Coverage": hybrid_coverage,
        "Personalization": hybrid_personalization,
        "Intra-list similarity": hybrid_ils,
        "TOP_N": TOP_N,
        "MAX_CATALOG_USERS": MAX_CATALOG_USERS,
    }
])

display(hybrid_eval_summary)

hybrid_eval_df[["UserId", "ProductId", "Score", "cf_score", "cb_score", "pop_score", "hybrid_score"]].to_csv(
    hybrid_dir / "hybrid_test_scores.csv", index=False
)
hybrid_topn_df.to_csv(hybrid_dir / "hybrid_topn.csv", index=False)
hybrid_eval_summary.to_csv(hybrid_dir / "hybrid_eval_summary.csv", index=False)
candidate_scores[["UserId", "ProductId", "cf_score", "cb_score", "pop_score", "hybrid_score"]].to_csv(
    hybrid_dir / "hybrid_candidate_scores.csv", index=False
)

print("Saved hybrid artifacts to:", hybrid_dir)

,Model,w_cf,w_cb,w_pop,RMSE,MAE,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage,Personalization,Intra-list similarity,TOP_N,MAX_CATALOG_USERS
0,Weighted Hybrid (CF+CB+Popularity),0.55,0.35,0.1,1.353068,1.12256,0.0,0.0,0.0,0.0,0.002591,0.052854,0.318596,10,300


Saved hybrid artifacts to: /Users/onosaeka/Uni/BCSAI/3-2/Chatbot&RecSys/Recommender-Sys/artifacts/hybrid
